# Example: Saving an equilibrium to geqdsk (via eqdsk)

In this example, we take a MAST-U-like equilibrium and show how to save it to geqdsk file via the [eqdsk](https://github.com/Fusion-Power-Plant-Framework/eqdsk) package (developed by the [Bluemira](https://github.com/Fusion-Power-Plant-Framework/bluemira) team). This is an alternative to the [FreeQDSK](https://github.com/freegs-plasma/FreeQDSK)-based approach shown in the previous example (03b) - `eqdsk` additionally understands COCOS (COordinate COnventions) and can also read/write JSON and IMAS.

As in the previous example, we run our standard equilibrium solve - making it lower single null by pushing the P6 current.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

# build machine
from freegsnke import build_machine
tokamak = build_machine.tokamak(
    active_coils_path=f"../machine_configs/MAST-U/MAST-U_like_active_coils.pickle",
    passive_coils_path=f"../machine_configs/MAST-U/MAST-U_like_passive_coils.pickle",
    limiter_path=f"../machine_configs/MAST-U/MAST-U_like_limiter.pickle",
    wall_path=f"../machine_configs/MAST-U/MAST-U_like_wall.pickle",
)

# initialise equilibrium object
from freegsnke import equilibrium_update
eq = equilibrium_update.Equilibrium(
    tokamak=tokamak,
    Rmin=0.1, Rmax=2.0,   # radial range
    Zmin=-2.2, Zmax=2.2,  # vertical range
    nx=65,                # number of grid points in the radial direction (needs to be of the form (2**n + 1) with n being an integer)
    ny=129,               # number of grid points in the vertical direction (needs to be of the form (2**n + 1) with n being an integer)
)

# initialise profile object
from freegsnke.jtor_update import ConstrainPaxisIp
profiles = ConstrainPaxisIp(
    eq=eq,
    paxis=8.1e3,
    Ip=6.2e5,
    fvac=0.5,
    alpha_m=1.8,
    alpha_n=1.2
)

# initialise solver
from freegsnke import GSstaticsolver
GSStaticSolver = GSstaticsolver.NKGSsolver(eq)

# set coil currents
import pickle
with open('data/simple_diverted_currents_PaxisIp.pk', 'rb') as f:
    current_values = pickle.load(f)

for key in current_values.keys():
    eq.tokamak[key].current = current_values[key]

# change this to shift plasma vertically up (-) or down (+)
eq.tokamak["P6"].current += 500

# carry out forward solve
GSStaticSolver.solve(eq=eq,
                     profiles=profiles,
                     constrain=None,
                     target_relative_tolerance=1e-9)

# plot the resulting equilibrium
fig1, ax1 = plt.subplots(1, 1, figsize=(4, 8), dpi=80)
ax1.grid(True, which='both')
eq.plot(axis=ax1, show=False)
eq.tokamak.plot(axis=ax1, show=False)
ax1.set_xlim(0.1, 2.15)
ax1.set_ylim(-2.25, 2.25)
plt.tight_layout()

## Writing a geqdsk file

`eqdsk`'s core class is `EQDSKInterface`, a dataclass holding all of the data in a geqdsk file. It's written to disk via its own `.write(path, file_format="geqdsk")` method.

Its field names differ slightly from FreeQDSK's (e.g. `nx`/`ny` &rarr; `nx`/`nz`, `rleft` &rarr; `xgrid1`, `simagx`/`sibdry` &rarr; `psimag`/`psibdry`, `cpasma` &rarr; `cplasma`, `rbdry`/`zbdry` &rarr; `xbdry`/`zbdry`, etc. - see the [package docs](https://github.com/Fusion-Power-Plant-Framework/eqdsk) for the full mapping). `EQDSKInterface` also carries a handful of required coil-geometry fields (`xc`/`zc`/`dxc`/`dzc`/`Ic`/`ncoil`) that plain geqdsk files don't otherwise need - FreeGSNKE tracks coil currents separately on the `tokamak` object, so we simply pass these through empty (`ncoil=0`).

In [ ]:
from eqdsk import EQDSKInterface

# grid quantities
nx, nz = eq.nx, eq.ny
R_1D, Z_1D = eq.R_1D, eq.Z_1D
rmin, rmax = R_1D[0], R_1D[-1]
zmin, zmax = Z_1D[0], Z_1D[-1]

# vacuum field
# bcentre = fvac / xcentre; xcentre is an arbitrary reference radius
# convention: often taken as the machine geometric centre or R at magnetic axis
xcentre = 0.5 * (rmin + rmax)         # assumed — check against your COCOS/convention
bcentre = profiles.fvac() / xcentre   # fvac passed into ConstrainPaxisIp as 0.5

# magnetic axis
xmag, zmag = eq.magneticAxis()[0:2]

# last closed flux surface (i.e. core plasma boundary)
sep = eq.separatrix(ntheta=180)        # (R, Z) points around LCFS
xbdry, zbdry = sep[:, 0], sep[:, 1]

# wall contour
xlim, zlim = eq.tokamak.wall.R, eq.tokamak.wall.Z

# 1D normalised psi
psi_n = eq.psiN_1D(N=nx)

# build the EQDSKInterface (see eqdsk docs for full field descriptions)
eqd = EQDSKInterface(
    name="MASTU_LSN",
    nx=nx,
    nz=nz,
    xdim=rmax - rmin,
    zdim=zmax - zmin,
    xcentre=xcentre,
    xgrid1=rmin,
    zmid=0.5 * (zmin + zmax),
    xmag=xmag,
    zmag=zmag,
    psimag=eq.psi_axis,
    psibdry=eq.psi_bndry,
    bcentre=bcentre,
    cplasma=eq.plasmaCurrent(),
    fpol=profiles.fpol(psi_n),
    pressure=profiles.pressure(psi_n),
    ffprime=profiles.ffprime(psi_n),
    pprime=profiles.pprime(psi_n),
    psi=eq.psi(),
    qpsi=eq.q(psi_n),
    nbdry=len(xbdry),
    xbdry=xbdry,
    zbdry=zbdry,
    nlim=len(xlim),
    xlim=xlim,
    zlim=zlim,
    # no coil geometry to store in a plain geqdsk file
    ncoil=0,
    xc=np.array([]),
    zc=np.array([]),
    dxc=np.array([]),
    dzc=np.array([]),
    Ic=np.array([]),
)

In [ ]:
# save the file!
eqd.write("MASTU_LSN.geqdsk", file_format="geqdsk")

## Reading a geqdsk file

`eqdsk` reads a geqdsk file back in via `EQDSKInterface.from_file(path)`, returning an `EQDSKInterface` instance whose fields are accessed as attributes rather than dictionary keys.

By default, `from_file` also tries to *identify* the file's COCOS and convert it to a target convention (COCOS 11) - useful when combining eqdsks written by different codes, but unnecessary here since we know exactly where this file came from. We pass `no_cocos=True` to skip that step and just get the raw values straight back.

A geqdsk file doesn't store individual coil currents, so - just as in the FreeQDSK example - we use the inverse solver to re-build the equilibrium, using the stored 2D psi map (and other data) as constraints.

We build the constraints entirely from what the geqdsk file gives us:

- an **isoflux constraint** from the last closed flux surface (`xbdry`/`zbdry`) - all these points should sit on the same flux contour, without us needing to know its absolute value;
- **null-point constraints** at the magnetic axis (`xmag`/`zmag`, also from the file);
- a **`psi_vals` constraint built from the *entire* saved 2D `psi` map** - unlike the constraints above, this one is given the actual flux *values*, not just which points should share a value, so it's what lets the optimiser pin down the absolute flux level (and, with it, the individual coil currents) rather than just the plasma shape.

To check that the file faithfully captures the equilibrium, we'll reconstruct it purely from what's in the geqdsk file: the grid extent, the plasma current (`cplasma`), and the `p'(ψ_n)`/`FF'(ψ_n)` profile data (`pprime`/`ffprime`).

FreeGSNKE's `GeneralPprimeFFprime` profile class accepts exactly this kind of tabulated profile data directly, rather than a parametric form like `ConstrainPaxisIp` above.

We reset the coil currents on a **fresh, independent** tokamak to zero first, to make sure we aren't just reusing the answer.

In [ ]:
# open the file!
eqd_read = EQDSKInterface.from_file("MASTU_LSN.geqdsk", no_cocos=True)

In [ ]:
from freegsnke.jtor_update import GeneralPprimeFFprime

# grid data
Rmin = eqd_read.xgrid1
Rmax = Rmin + eqd_read.xdim
Zmin = eqd_read.zmid - 0.5 * eqd_read.zdim
Zmax = eqd_read.zmid + 0.5 * eqd_read.zdim
nx_read, ny_read = eqd_read.nx, eqd_read.nz

# normalised psi
psi_n_read = np.linspace(0, 1, nx_read)

Now let's set up and run the inverse solver, using those quantities plus the constraints described above to reconstruct the coil currents.

In [ ]:
from freegsnke.inverse import Inverse_optimizer

# coil names
active_coil_names = tokamak.coils_list[: tokamak.n_active_coils]

# fresh tokamak so we can set coil currents to zero and compare to old eq object
tokamak_inv = build_machine.tokamak(
    active_coils_path=f"../machine_configs/MAST-U/MAST-U_like_active_coils.pickle",
    passive_coils_path=f"../machine_configs/MAST-U/MAST-U_like_passive_coils.pickle",
    limiter_path=f"../machine_configs/MAST-U/MAST-U_like_limiter.pickle",
    wall_path=f"../machine_configs/MAST-U/MAST-U_like_wall.pickle",
)

# fresh equilibrium object
eq_inv = equilibrium_update.Equilibrium(
    tokamak=tokamak_inv,
    Rmin=Rmin, Rmax=Rmax,
    Zmin=Zmin, Zmax=Zmax,
    nx=nx_read,
    ny=ny_read,
)

# fresh profiles object
profiles_inv = GeneralPprimeFFprime(
    eq=eq_inv,
    Ip=eqd_read.cplasma,
    fvac=eqd_read.bcentre * eqd_read.xcentre,
    psi_n=psi_n_read,
    pprime_data=eqd_read.pprime,
    ffprime_data=eqd_read.ffprime,
)

# isoflux constraint: subsample the LCFS boundary stored in the geqdsk file
isoflux_set = [[eqd_read.xbdry[::5], eqd_read.zbdry[::5]]]

# null-point constraints: magnetic axis (from the file)
null_points = [
    [eqd_read.xmag],
    [eqd_read.zmag],
]

# psi_vals constraint built from the *entire* saved 2D psi map. Passing the grid's own
# (R, Z) meshgrids (rather than a sparse list of points) triggers Inverse_optimizer's
# "full grid" fast path, which reuses the equilibrium's own cached Greens functions
# instead of recomputing them pointwise.
psi_vals = [eq_inv.R, eq_inv.Z, eqd_read.psi]

# build constraints
constrain = Inverse_optimizer(
    isoflux_set=isoflux_set,
    null_points=null_points,
    psi_vals=psi_vals,
)

# initialise solver object
GSStaticSolver_inv = GSstaticsolver.NKGSsolver(eq_inv)

In [ ]:
# set coils to zero current
for label in active_coil_names:
    eq_inv.tokamak.set_coil_current(label, 0.0)

# default plasma flux guess
eq_inv.plasma_psi = eq_inv.create_psi_plasma_default()

# solve
GSStaticSolver_inv.solve(
    eq=eq_inv,
    profiles=profiles_inv,
    constrain=constrain,
    target_relative_tolerance=1e-4,
    target_relative_psit_update=1e-3,
    max_solving_iterations=50,
    l2_reg=1e-9,
    verbose=True,
    Picard_handover=1e-5,
    full_jacobian_handover=[2e-2,1e-2],
)

This converges properly and gives the correct total flux, coil currents close to their true values, and `psi_axis`/`psi_bndry` matching to several decimal places.

In [ ]:
# print coil currents
reconstructed_currents = {label: eq_inv.tokamak[label].current for label in active_coil_names}
original_active_currents = {label: eq.tokamak[label].current for label in active_coil_names}

print("Coil currents [A]:")
print(f"{'coil':>10s}  {'original':>12s}  {'reconstructed':>14s}")
for label in active_coil_names:
    print(f"{label:>10s}  {original_active_currents[label]:12.1f}  {reconstructed_currents[label]:14.1f}")

# some small differences
psi_diff_inv = np.abs(eq.psi() - eq_inv.psi())
print(f"\nMagnetic axis:   original = {eq.magneticAxis()[0:2]}, reconstructed = {eq_inv.magneticAxis()[0:2]}")
print(f"psi_axis:        original = {eq.psi_axis:.6g}, reconstructed = {eq_inv.psi_axis:.6g}")
print(f"psi_boundary:    original = {eq.psi_bndry:.6g}, reconstructed = {eq_inv.psi_bndry:.6g}")
print(f"Max |Δpsi| relative to max|psi|: {np.max(psi_diff_inv) / np.max(np.abs(eq.psi())):.3e}")

# plot the two equilibria side by side
fig1, (ax1, ax2) = plt.subplots(1, 2, figsize=(8, 8), dpi=80)
for ax, e, title in [(ax1, eq, "Original"), (ax2, eq_inv, "Inverse reconstruction")]:
    ax.grid(True, which="both")
    e.plot(axis=ax, show=False)
    e.tokamak.plot(axis=ax, show=False)
    ax.set_xlim(0.1, 2.15)
    ax.set_ylim(-2.25, 2.25)
    ax.set_title(title)
    if ax == ax2:
        constrain.plot(axis=ax2)
plt.tight_layout()